In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql.window import Window
from pyspark.sql import functions as F

In [0]:
df = spark.read.table('fintech360.data_quality_passed.bank_data')
df.limit(5).display()

In [0]:

from pyspark.sql import functions as F

# ============================================================
# 1. READ DATA QUALITY PASSED BANK DATA
# ============================================================

source_df = spark.read.table(
    "fintech360.data_quality_passed.bank_data"
)

# ============================================================
# 2. STANDARDIZE BUSINESS DIMENSIONS
# ============================================================

df = (source_df
    .withColumn(
        "business_bank_name",
        F.upper(F.trim(F.col("bank_name")))
    )
    .withColumn(
        "business_transaction_type",
        F.upper(F.trim(F.col("transaction_type")))
    )
    .withColumn(
        "business_transaction_category",
        F.upper(F.trim(F.col("transaction_category")))
    )
    .withColumn(
        "business_currency",
        F.upper(F.trim(F.col("currency")))
    )
    .withColumn(
        "business_narration",
        F.upper(F.trim(F.col("narration")))
    )
    .withColumn(
        "business_transaction_amount",
        F.col("transaction_amount").cast("double")
    )
    .withColumn(
        "business_signed_amount",
        F.col("signed_amount").cast("double")
    )
    .withColumn(
        "business_balance",
        F.col("balance_after_txn").cast("double")
    )
    .withColumn(
        "business_salary_flag",
        F.col("is_salary_like").cast("int")
    )
)

# ============================================================
# 3. DERIVE TRANSACTION DATE AND TIME DIMENSIONS
# Unix timestamps are in seconds.
# ============================================================

df = (df
    .withColumn(
        "business_transaction_datetime",
        F.to_timestamp(
            F.from_unixtime(
                F.col("transaction_ts").cast("long")
            )
        )
    )
    .withColumn(
        "business_transaction_date",
        F.to_date("business_transaction_datetime")
    )
    .withColumn(
        "business_transaction_month",
        F.date_format(
            "business_transaction_datetime", "yyyy-MM"
        )
    )
    .withColumn(
        "business_transaction_year",
        F.year("business_transaction_datetime")
    )
    .withColumn(
        "business_transaction_day",
        F.dayofmonth("business_transaction_datetime")
    )
    .withColumn(
        "business_day_of_week",
        F.date_format(
            "business_transaction_datetime", "EEEE"
        )
    )
)

# ============================================================
# 4. CASH INFLOW AND OUTFLOW
# ============================================================

df = (df
    .withColumn(
        "cash_inflow",
        F.when(
            F.col("business_transaction_type") == "CREDIT",
            F.col("business_transaction_amount")
        ).otherwise(F.lit(0.0))
    )
    .withColumn(
        "cash_outflow",
        F.when(
            F.col("business_transaction_type") == "DEBIT",
            F.col("business_transaction_amount")
        ).otherwise(F.lit(0.0))
    )
    .withColumn(
        "net_cash_flow",
        F.col("cash_inflow") - F.col("cash_outflow")
    )
)

# ============================================================
# 5. SALARY BUSINESS FEATURES
# A salary-like flag alone does not confirm salary.
# Identify flagged credits separately from flagged debits.
# ============================================================

df = (df
    .withColumn(
        "salary_like_credit",
        F.when(
            (F.col("business_salary_flag") == 1) &
            (F.col("business_transaction_type") == "CREDIT"),
            F.lit(1)
        ).otherwise(F.lit(0))
    )
    .withColumn(
        "salary_like_debit_flag",
        F.when(
            (F.col("business_salary_flag") == 1) &
            (F.col("business_transaction_type") == "DEBIT"),
            F.lit(1)
        ).otherwise(F.lit(0))
    )
    .withColumn(
        "salary_like_credit_amount",
        F.when(
            F.col("salary_like_credit") == 1,
            F.col("business_transaction_amount")
        ).otherwise(F.lit(0.0))
    )
    .withColumn(
        "salary_narration_flag",
        F.when(
            F.col("business_narration").rlike(
                r"\b(SALARY|PAYROLL|WAGES)\b"
            ),
            F.lit(1)
        ).otherwise(F.lit(0))
    )
    .withColumn(
        "salary_signal_status",
        F.when(
            (F.col("salary_like_credit") == 1) &
            (F.col("salary_narration_flag") == 1),
            F.lit("FLAGGED_CREDIT_WITH_SALARY_NARRATION")
        ).when(
            F.col("salary_like_credit") == 1,
            F.lit("FLAGGED_CREDIT_REQUIRES_VERIFICATION")
        ).when(
            F.col("salary_like_debit_flag") == 1,
            F.lit("FLAGGED_DEBIT_REQUIRES_REVIEW")
        ).when(
            F.col("salary_narration_flag") == 1,
            F.lit("SALARY_NARRATION_REQUIRES_REVIEW")
        ).otherwise(F.lit("NO_SALARY_SIGNAL"))
    )
)

# ============================================================
# 6. TRANSACTION AMOUNT BANDS
# ============================================================

df = df.withColumn(
    "transaction_amount_band",
    F.when(
        F.col("business_transaction_amount") < 500,
        "01_BELOW_500"
    ).when(
        F.col("business_transaction_amount") < 2000,
        "02_500_TO_1999"
    ).when(
        F.col("business_transaction_amount") < 5000,
        "03_2000_TO_4999"
    ).when(
        F.col("business_transaction_amount") < 10000,
        "04_5000_TO_9999"
    ).otherwise("05_10000_PLUS")
)

# ============================================================
# 7. BALANCE BANDS
# ============================================================

df = df.withColumn(
    "balance_band",
    F.when(
        F.col("business_balance") < 1000,
        "01_LOW_BALANCE"
    ).when(
        F.col("business_balance") < 5000,
        "02_1000_TO_4999"
    ).when(
        F.col("business_balance") < 10000,
        "03_5000_TO_9999"
    ).when(
        F.col("business_balance") < 25000,
        "04_10000_TO_24999"
    ).otherwise("05_25000_PLUS")
)

# ============================================================
# 8. BUSINESS TRANSACTION CLASSIFICATION
# Keep source category; derive a separate narration-based signal.
# ============================================================

df = df.withColumn(
    "narration_business_type",
    F.when(
        F.col("business_narration").rlike(
            r"\b(ATM|CASH WITHDRAWAL)\b"
        ),
        "CASH_WITHDRAWAL"
    ).when(
        F.col("business_narration").rlike(
            r"\b(UPI|IMPS|NEFT|RTGS|TRANSFER)\b"
        ),
        "MONEY_TRANSFER"
    ).when(
        F.col("business_narration").rlike(
            r"\b(SALARY|PAYROLL|WAGES)\b"
        ),
        "SALARY_RELATED"
    ).when(
        F.col("business_narration").rlike(
            r"\b(RENT)\b"
        ),
        "RENT_RELATED"
    ).when(
        F.col("business_narration").rlike(
            r"\b(CARD PAYMENT|CREDIT CARD)\b"
        ),
        "CARD_PAYMENT"
    ).otherwise("OTHER_OR_UNCLASSIFIED")
)

# ============================================================
# 9. BUSINESS REVIEW FLAGS
# These are analytical signals, not proof of fraud or risk.
# ============================================================

df = (df
    .withColumn(
        "low_balance_flag",
        F.when(
            F.col("business_balance") < 2000,
            F.lit(1)
        ).otherwise(F.lit(0))
    )
    .withColumn(
        "high_value_transaction_flag",
        F.when(
            F.col("business_transaction_amount") >= 10000,
            F.lit(1)
        ).otherwise(F.lit(0))
    )
    .withColumn(
        "transaction_direction",
        F.when(
            F.col("business_transaction_type") == "CREDIT",
            "INFLOW"
        ).when(
            F.col("business_transaction_type") == "DEBIT",
            "OUTFLOW"
        ).otherwise("UNKNOWN")
    )
)

# ============================================================
# 10. ADD TRANSFORMATION METADATA
# ============================================================

df = df.withColumn(
    "business_transformed_at",
    F.current_timestamp()
)

# ============================================================
# 11. FINAL BUSINESS DATAFRAME
# ============================================================

bank_data_business = df

display(bank_data_business.limit(10))


In [0]:

# Create destination schema
spark.sql("""
CREATE SCHEMA IF NOT EXISTS fintech360.business_transformation
""")

# Append transformed bank transactions
(bank_data_business.write
    .format("delta")
    .mode("append")
    .saveAsTable("fintech360.business_transformation.bank_data"))

print("Business transformation completed.")
print("Target: fintech360.business_transformation.bank_data")
